# STEP 2: Audio Cleaning Pipeline
Picks up from `dataset_segmented.csv` produced by the VAD/segmentation step.

Pipeline per segment:
1. Load WAV
2. Voice / background removal (noisereduce spectral gating)
3. Bandpass filter + spectral subtraction denoising
4. Amplitude normalisation
5. Fixed-length pad/trim  →  0.5 s for cough, 3.0 s for vowel
6. Save cleaned WAVs + updated CSV

In [ ]:
# ── Install voice-removal library if not already present ──────────────────────
# Run once, then comment out
# !pip install noisereduce

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import noisereduce as nr
from scipy.signal import butter, filtfilt
from tqdm import tqdm

# ── Paths ──────────────────────────────────────────────────────────────────────
SEGMENTED_CSV = "/Volumes/Samsung_T5/respiratory_classification_project/dataset_segmented.csv"
CLEAN_DIR     = "/Volumes/Samsung_T5/respiratory_classification_project/cleaned"
CLEAN_CSV     = "/Volumes/Samsung_T5/respiratory_classification_project/dataset_cleaned.csv"

# ── Audio constants ────────────────────────────────────────────────────────────
SR            = 22050
COUGH_LEN_SEC = 0.5    # fixed length for cough segments
VOWEL_LEN_SEC = 3.0    # fixed length for vowel segments
N_FFT         = 2048
HOP_LENGTH    = 512

os.makedirs(CLEAN_DIR, exist_ok=True)
print("Libraries loaded ✅")

## Step 1 — Voice / Background Removal
Uses `noisereduce` spectral gating.  
Takes the first 200 ms as the noise profile (mic hum, room tone, residual speech harmonics) and subtracts it.  
`stationary=False` handles time-varying noise like a patient speaking before coughing.

In [ ]:
def remove_voice_noise(y, sr):
    noise_sample_len = int(0.2 * sr)          # first 200ms as noise reference
    noise_sample     = y[:noise_sample_len]

    y_reduced = nr.reduce_noise(
        y             = y,
        sr            = sr,
        y_noise       = noise_sample,          # explicit noise profile
        prop_decrease = 0.9,                   # reduce noise by 90%
        stationary    = False,                 # handles non-stationary background voice
        n_fft         = N_FFT,
        hop_length    = HOP_LENGTH
    )
    return y_reduced

## Step 2 — Denoising (Bandpass + Spectral Subtraction)
Bandpass filter keeps 100 Hz – 8 kHz (the respiratory frequency range).  
Spectral subtraction removes residual broadband noise using the first 10 STFT frames as a noise estimate.

In [ ]:
def denoise_audio(y, sr):
    # DC offset removal
    y = y - np.mean(y)

    # High-pass at 100 Hz  →  removes rumble / very low hum
    b, a = butter(4, 100 / (sr / 2), btype='high')
    y = filtfilt(b, a, y)

    # Low-pass at 8 kHz  →  removes high-frequency hiss
    b, a = butter(4, 8000 / (sr / 2), btype='low')
    y = filtfilt(b, a, y)

    # Spectral subtraction
    S         = librosa.stft(y)
    magnitude = np.abs(S)
    phase     = np.angle(S)
    noise_est = np.mean(magnitude[:, :10], axis=1, keepdims=True)  # first 10 frames = noise
    mag_clean = np.maximum(magnitude - noise_est, 0)
    S_clean   = mag_clean * np.exp(1j * phase)
    y_clean   = librosa.istft(S_clean)

    # Normalise amplitude to [-1, 1]
    y_clean = y_clean / (np.max(np.abs(y_clean)) + 1e-9)
    return y_clean

## Step 3 — Fixed-Length Pad / Trim
Coughs → 0.5 s, Vowels → 3.0 s.  
Clips longer than the target are trimmed from the end; shorter ones are zero-padded at the end.

In [ ]:
def fix_length(y, sr, audio_type):
    target_sec     = COUGH_LEN_SEC if audio_type == "cough" else VOWEL_LEN_SEC
    target_samples = int(target_sec * sr)

    if len(y) >= target_samples:
        return y[:target_samples]                              # trim
    else:
        return np.pad(y, (0, target_samples - len(y)),
                      mode='constant')                         # zero-pad at end

## Step 4 — Full Cleaning Pipeline
Chains all three steps: load → voice removal → denoising → fix length → save.

In [ ]:
def clean_and_save(path, audio_type, clean_dir):
    """
    Cleans one segment and saves the result to clean_dir.
    Returns the output path, or None on failure.
    """
    try:
        y, sr = librosa.load(path, sr=SR, mono=True)
    except Exception as e:
        print(f"  ❌ Load failed: {path} | {e}")
        return None

    # 1. Voice / background removal
    y = remove_voice_noise(y, sr)

    # 2. Bandpass + spectral denoising
    y = denoise_audio(y, sr)

    # 3. Fixed-length pad / trim
    y = fix_length(y, sr, audio_type)

    # 4. Save
    fname    = os.path.basename(path)
    out_path = os.path.join(clean_dir, fname)
    sf.write(out_path, y, sr)

    return out_path

## Step 5 — Run on All Segments

In [ ]:
df_seg = pd.read_csv(SEGMENTED_CSV)
print(f"Loaded {len(df_seg)} segments")
print(df_seg["audio_type"].value_counts())
print(df_seg["disease"].value_counts())

In [ ]:
clean_paths = []
failed      = 0

for _, row in tqdm(df_seg.iterrows(), total=len(df_seg), desc="Cleaning segments"):
    path       = row["segment_path"]
    audio_type = row["audio_type"]

    if not os.path.exists(str(path)):
        clean_paths.append(None)
        failed += 1
        continue

    out_path = clean_and_save(path, audio_type, CLEAN_DIR)
    clean_paths.append(out_path)
    if out_path is None:
        failed += 1

# Add cleaned path column and drop failed rows
df_seg["clean_path"] = clean_paths
df_clean = df_seg.dropna(subset=["clean_path"])
df_clean.to_csv(CLEAN_CSV, index=False)

print(f"\n✅ Done!")
print(f"   Total segments   : {len(df_seg)}")
print(f"   Cleaned & saved  : {len(df_clean)}")
print(f"   Failed / skipped : {failed}")
print(f"   Saved to         : {CLEAN_CSV}")

## Sanity Check — Before vs After for One Segment

In [ ]:
import matplotlib.pyplot as plt

# Pick first cough segment
test_row   = df_clean[df_clean["audio_type"] == "cough"].iloc[0]
raw_path   = test_row["segment_path"]
clean_path = test_row["clean_path"]

y_raw,   sr = librosa.load(raw_path,   sr=SR, mono=True)
y_clean, sr = librosa.load(clean_path, sr=SR, mono=True)

fig, axes = plt.subplots(2, 2, figsize=(16, 8))
fig.suptitle(f"Before vs After Cleaning — {test_row['disease']} cough", fontsize=13)

# Raw waveform
axes[0, 0].plot(np.linspace(0, len(y_raw)/sr, len(y_raw)), y_raw, alpha=0.7)
axes[0, 0].set_title("Waveform — RAW")
axes[0, 0].set_xlabel("Time (s)")
axes[0, 0].set_ylabel("Amplitude")

# Cleaned waveform
axes[0, 1].plot(np.linspace(0, len(y_clean)/sr, len(y_clean)), y_clean, color='green', alpha=0.7)
axes[0, 1].set_title("Waveform — CLEANED")
axes[0, 1].set_xlabel("Time (s)")
axes[0, 1].set_ylabel("Amplitude")

# Raw spectrogram
D_raw = librosa.amplitude_to_db(np.abs(librosa.stft(y_raw)), ref=np.max)
librosa.display.specshow(D_raw, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 0])
axes[1, 0].set_title("Spectrogram — RAW")
axes[1, 0].set_ylim(0, 8000)

# Cleaned spectrogram
D_clean = librosa.amplitude_to_db(np.abs(librosa.stft(y_clean)), ref=np.max)
img = librosa.display.specshow(D_clean, sr=sr, x_axis='time', y_axis='hz', ax=axes[1, 1])
axes[1, 1].set_title("Spectrogram — CLEANED")
axes[1, 1].set_ylim(0, 8000)
fig.colorbar(img, ax=axes[1, 1], format='%+2.0f dB')

plt.tight_layout()
plt.show()